#### Step 3: Model Fine-tuning
In this notebook, you'll fine-tune the Meta Llama 2 7B large language model, deploy the fine-tuned model, and test it's text generation and domain knowledge capabilities. 

Fine-tuning refers to the process of taking a pre-trained language model and retraining it for a different but related task using specific data. This approach is also known as transfer learning, which involves transferring the knowledge learned from one task to another. Large language models (LLMs) like Llama 2 7B are trained on massive amounts of unlabeled data and can be fine-tuned on domain domain datasets, making the model perform better on that specific domain.

Input: A train and an optional validation directory. Each directory contains a CSV/JSON/TXT file.
For CSV/JSON files, the train or validation data is used from the column called 'text' or the first column if no column called 'text' is found.
The number of files under train and validation should equal to one.

- **You'll choose your dataset below based on the domain you've chosen**

Output: A trained model that can be deployed for inference.\
After you've fine-tuned the model, you'll evaluate it with the same input you used in project step 2: model evaluation. 

---

#### Set up

---
Install and import the necessary packages. Restart the kernel after executing the cell below. 

---

In [ ]:
import sys

# Reset SageMaker SDK packages, then use the stable 2.x JumpStart API path.
# After this cell finishes, restart the kernel before running the cells below.
!{sys.executable} -m pip install -U pip --quiet
!{sys.executable} -m pip uninstall -y \
    sagemaker \
    sagemaker-core \
    sagemaker-train \
    sagemaker-serve \
    sagemaker-mlops \
    --quiet 2>/dev/null
!{sys.executable} -m pip install --no-cache-dir \
    "sagemaker>=2.231.0,<3" \
    datasets \
    ipywidgets \
    --quiet


Select the model to fine-tune

In [ ]:
# Pin an exact JumpStart model version to avoid forward-compatibility warnings
# and keep EULA/input/output signatures stable across notebook reruns.
model_id = "meta-textgeneration-llama-2-7b"
model_version = "2.1.8"


In the cell below, choose the training dataset text for the domain you've chosen and update the code in the cell below:  

To create a finance domain expert model: 

- `"training": f"s3://genaiwithawsproject2024/training-datasets/finance"`

To create a medical domain expert model: 

- `"training": f"s3://genaiwithawsproject2024/training-datasets/medical"`

To create an IT domain expert model: 

- `"training": f"s3://genaiwithawsproject2024/training-datasets/it"`

In [ ]:
import logging
import warnings

import boto3
import sagemaker
from sagemaker.session import get_execution_role
from sagemaker.jumpstart.estimator import JumpStartEstimator

logging.getLogger("sagemaker.telemetry.telemetry_logging").setLevel(logging.WARNING)
warnings.filterwarnings(
    "ignore",
    message=r"For forward compatibility, pin to model_version=.*",
    category=UserWarning,
)

boto_session = boto3.Session()
sess = sagemaker.Session(boto_session=boto_session)

try:
    role = get_execution_role(sess)
except Exception:
    identity = boto_session.client("sts").get_caller_identity()
    caller_arn = identity["Arn"]
    account_id = identity["Account"]
    if ":assumed-role/" in caller_arn:
        role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
        role = f"arn:aws:iam::{account_id}:role/{role_name}"
    else:
        role = caller_arn

# Choose one: "finance", "medical", or "it".
domain = "finance"
training_datasets = {
    "finance": "s3://genaiwithawsproject2024/training-datasets/finance",
    "medical": "s3://genaiwithawsproject2024/training-datasets/medical",
    "it": "s3://genaiwithawsproject2024/training-datasets/it",
}
training_dataset_s3_uri = training_datasets[domain]

estimator = JumpStartEstimator(
    model_id=model_id,
    model_version=model_version,
    role=role,
    sagemaker_session=sess,
    instance_type="ml.g5.2xlarge",
    environment={"accept_eula": "true"},
)

estimator.set_hyperparameters(instruction_tuned="False", epoch="5")

print("Role:", role)
print("Training dataset:", training_dataset_s3_uri)
estimator.fit({"training": training_dataset_s3_uri})


#### Deploy the fine-tuned model
---
Next, we deploy the domain fine-tuned model. We will compare the performance of the fine-tuned and pre-trained model.

---

In [ ]:
import boto3
import uuid

sm_client = boto3.client("sagemaker")
finetuned_endpoint_name = f"llama2-ft-{domain}-{uuid.uuid4().hex[:8]}"

# Explicitly set the hosting instance type so SageMaker does not choose
# a larger default instance than the project environment allows.
try:
    try:
        finetuned_predictor = estimator.deploy(
            instance_type="ml.g5.2xlarge",
            initial_instance_count=1,
            endpoint_name=finetuned_endpoint_name,
            accept_eula=True,
        )
    except TypeError:
        finetuned_predictor = estimator.deploy(
            instance_type="ml.g5.2xlarge",
            initial_instance_count=1,
            endpoint_name=finetuned_endpoint_name,
        )
except Exception as e:
    print(f"Deployment failed for endpoint: {finetuned_endpoint_name}")
    print(e)
    try:
        details = sm_client.describe_endpoint(EndpointName=finetuned_endpoint_name)
        print("Status:", details.get("EndpointStatus"))
        print("FailureReason:", details.get("FailureReason", ""))
    except Exception as describe_error:
        print("Could not describe failed endpoint:", describe_error)
    raise


#### Evaluate the pre-trained and fine-tuned model
---
Next, we use the same input from the model evaluation step to evaluate the performance of the fine-tuned model and compare it with the base pre-trained model. 

---

Create a function to print the response from the model

In [ ]:
def print_response(payload, response):
    if isinstance(response, list) and response:
        text = response[0].get("generation") or response[0].get("generated_text") or response[0]
    elif isinstance(response, dict):
        text = response.get("generation") or response.get("generated_text") or response
    else:
        text = response

    print(payload["inputs"])
    print(f"> {text}")
    print("\n==================================\n")


Now we can run the same prompts on the fine-tuned model to evaluate it's domain knowledge.   

**Replace "inputs"** in the next cell with the input to send the model based on the domain you've chosen. 

**For financial domain:**

  "inputs": "Replace with sentence below from text"  
- "The  investment  tests  performed  indicate"
- "the  relative  volume  for  the  long  out  of  the  money  options, indicates"
- "The  results  for  the  short  in  the  money  options"
- "The  results  are  encouraging  for  aggressive  investors"

**For medical domain:** 

  "inputs": "Replace with sentence below from text" 
- "Myeloid neoplasms and acute leukemias derive from"
- "Genomic characterization is essential for"
- "Certain germline disorders may be associated with"
- "In contrast to targeted approaches, genome-wide sequencing"

**For IT domain:** 

  "inputs": "Replace with sentence below from text" 
- "Traditional approaches to data management such as"
- "A second important aspect of ubiquitous computing environments is"
- "because ubiquitous computing is intended to" 
- "outline the key aspects of ubiquitous computing from a data management perspective."

In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

prompt_inputs = {
    "finance": "The investment tests performed indicate",
    "medical": "Genomic characterization is essential for",
    "it": "Traditional approaches to data management such as",
}

payload = {
    "inputs": prompt_inputs[domain],
    "parameters": {
        "max_new_tokens": 64,
        "top_p": 0.9,
        "temperature": 0.6,
        "return_full_text": False,
    },
}

finetuned_predictor.serializer = JSONSerializer()
finetuned_predictor.deserializer = JSONDeserializer()

try:
    try:
        response = finetuned_predictor.predict(payload, custom_attributes="accept_eula=true")
    except TypeError:
        response = finetuned_predictor.predict(payload)
    print_response(payload, response)
except Exception as e:
    print(e)


Do the outputs from the fine-tuned model provide domain-specific insightful and relevant content? You can continue experimenting with the inputs of the model to test it's domain knowledge. 

**Use the output from this notebook to fill out the "model fine-tuning" section of the project documentation report**

**After you've filled out the report, run the cells below to delete the model deployment** 

`IF YOU FAIL TO RUN THE CELLS BELOW YOU WILL RUN OUT OF BUDGET TO COMPLETE THE PROJECT`

In [ ]:
import boto3
from botocore.exceptions import ClientError

sm_client = boto3.client("sagemaker")
endpoint_config_name = None
model_names = []

try:
    endpoint_details = sm_client.describe_endpoint(EndpointName=finetuned_endpoint_name)
    endpoint_config_name = endpoint_details.get("EndpointConfigName")
except ClientError as e:
    print(f"Endpoint already deleted or unavailable: {e.response['Error'].get('Message', e)}")

if endpoint_config_name:
    try:
        endpoint_config = sm_client.describe_endpoint_config(
            EndpointConfigName=endpoint_config_name
        )
        model_names = [
            variant["ModelName"]
            for variant in endpoint_config.get("ProductionVariants", [])
            if "ModelName" in variant
        ]
    except ClientError as e:
        print(f"Endpoint config already deleted or unavailable: {e.response['Error'].get('Message', e)}")

try:
    sm_client.delete_endpoint(EndpointName=finetuned_endpoint_name)
    print(f"Deleted endpoint: {finetuned_endpoint_name}")
except ClientError as e:
    print(f"Endpoint delete skipped: {e.response['Error'].get('Message', e)}")

if endpoint_config_name:
    try:
        sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
        print(f"Deleted endpoint config: {endpoint_config_name}")
    except ClientError as e:
        print(f"Endpoint config delete skipped: {e.response['Error'].get('Message', e)}")

for model_name in model_names:
    try:
        sm_client.delete_model(ModelName=model_name)
        print(f"Deleted model: {model_name}")
    except ClientError as e:
        print(f"Model delete skipped for {model_name}: {e.response['Error'].get('Message', e)}")
